# [Pipeline Title] — Predictive Model

---

## 1. Problem Framing

### Business Problem

<!-- TODO: Describe the business problem in plain language. Who is affected?
What decision is currently made without data? What does this pipeline enable? -->

This pipeline answers the question: **[TODO: one sentence business question]**

The deployed output is a **[TODO: score / flag]** surfaced on
[TODO: where in the application — case dashboard / social media tool / donor dashboard].

### Who Cares About This

- **[Role]** — [why they care and how they use the output]
- **[Role]** — [why they care and how they use the output]

### Predictive vs. Explanatory

This pipeline uses a **predictive approach**. The goal is to generate reliable
scores for individual [residents / posts / donors] on new, unseen data. Out-of-sample
performance is the primary success criterion — interpretability is secondary.

<!-- TODO: One sentence on what the model cannot do (causal claims, edge cases) -->

### Success Metrics

- **Primary:** [ROC-AUC / R²] — [one sentence justification]
- **Secondary:** [F1, Balanced Accuracy / RMSE, MAE]
- **Operational threshold:** [TODO: which error is worse and why]

### A Note on Training Data

<!-- TODO: Note any known limitations of the training data — duplication,
sample size, domain shift between sample and production data. -->


---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

sys.path.append(os.path.dirname(os.path.abspath('.')))
os.chdir('..')

# TODO: Import the correct prepare function
# from functions.fn_domain_prep import prepare_residents
# from functions.fn_domain_prep import prepare_donors
# from functions.fn_domain_prep import prepare_social_media

from functions.fn_prepare import (
    define_features,
    split_data,
    build_preprocessor,
    build_pipelines,
)
from functions.fn_model_predict import (
    run_cross_validation,
    tune_model,
    evaluate_final_model,
    save_model,
)

print("All imports successful.")

### 2.1 Load and Prepare Data

`prepare_[domain]()` encodes every cleaning and feature engineering decision from
`eda_[domain].ipynb` into a single reproducible call. It tries Azure SQL first
(via `appsettings.Development.json`) and falls back to the local `data/` CSVs.

**Tables joined:** `[TODO: list tables]`

**Key preparation decisions encoded:**
- Structural columns dropped: [TODO: list]
- [TODO: date features engineered]
- [TODO: intentional nulls handled]
- [TODO: aggregations performed]
- `[target]` engineered from `[source column / logic]`

In [ ]:
# TODO: Call the correct prepare function
# df, NUMERIC, CATEGORICAL, DROP = prepare_residents()
# df, NUMERIC, CATEGORICAL, DROP = prepare_donors()
# df, NUMERIC, CATEGORICAL, DROP = prepare_social_media()

TARGET = '[TODO: target column name]'

print(f"Dataset shape: {df.shape}")
print(f"Target: '{TARGET}'")
# Classification: print(f"Base rate: {df[TARGET].mean():.1%} positive")
# Regression:     print(f"Mean: {df[TARGET].mean():.2f}  |  Std: {df[TARGET].std():.2f}")

### 2.2 Feature Definition

`define_features()` is called with `DROP['[target]']` — the per-target drop list
returned by `prepare_[domain]()` and defined during EDA. It prevents:

- **Direct leakage:** [TODO: column that directly encodes the target]
- **Cross-target contamination:** other pipeline targets excluded
- **[TODO: any domain-specific leakage for this target]**

In [ ]:
X, y = define_features(
    df,
    target=TARGET,
    numeric=NUMERIC,
    categorical=CATEGORICAL,
    drop_cols=DROP[TARGET],
)

categorical_in_X = [c for c in CATEGORICAL if c in X.columns]
numeric_in_X     = [c for c in NUMERIC     if c in X.columns]
X[categorical_in_X] = X[categorical_in_X].astype(str).replace({'nan': np.nan, '<NA>': np.nan})

print(f"Feature matrix: {X.shape[0]} rows × {X.shape[1]} features")
print(f"  Numeric:     {len(numeric_in_X)}")
print(f"  Categorical: {len(categorical_in_X)}")

### 2.3 Exploratory Confirmation

EDA was conducted in `eda_[domain].ipynb`. The cells below confirm that expected
signals are present in the prepared feature matrix — documentation, not new decisions.

In [ ]:
# Top numeric features by absolute correlation with target
corr = X[numeric_in_X].corrwith(y).abs().sort_values(ascending=False)
print(f"Top 10 numeric features by |correlation| with {TARGET}:")
print(corr.head(10).round(3).to_string())

In [ ]:
# TODO: Categorical breakdown — replace col names
# for col in ['[key_cat_1]', '[key_cat_2]']:
#     if col in X.columns:
#         rate = (pd.concat([X[col], y], axis=1)
#                   .groupby(col)[TARGET]
#                   .agg(['mean', 'count'])
#                   .rename(columns={'mean': 'rate', 'count': 'n'})
#                   .sort_values('rate', ascending=False))
#         print(f"\n{TARGET} rate by {col}:")
#         print(rate.round(3).to_string())

In [ ]:
# TODO: Distribution plots — replace feature names
# fig, axes = plt.subplots(1, 3, figsize=(14, 4))
# features_to_plot = ['[feat_1]', '[feat_2]', '[feat_3]']
#
# Classification:
# for ax, feat in zip(axes, features_to_plot):
#     if feat not in X.columns: continue
#     for val, label in {0: 'Negative', 1: 'Positive'}.items():
#         ax.hist(X.loc[y == val, feat], alpha=0.6, label=label, bins=20)
#     ax.set_title(feat); ax.set_xlabel('Value'); ax.legend()
# plt.suptitle('Key Feature Distributions by Target', y=1.02)
# plt.tight_layout(); plt.show()
#
# Regression:
# for ax, feat in zip(axes, features_to_plot):
#     ax.scatter(X[feat], y, alpha=0.4, s=15)
#     ax.set_xlabel(feat); ax.set_ylabel(TARGET)
# plt.tight_layout(); plt.show()

---
## 3. Modeling & Feature Selection

### 3.1 Train/Test Split

The test set is locked here and not touched again until Section 4.

In [ ]:
PROBLEM_TYPE = '[classification / regression]'
stratify     = (PROBLEM_TYPE == 'classification')

X_train, X_test, y_train, y_test = split_data(X, y, stratify=stratify)

### 3.2 Candidate Model Comparison

<!-- TODO: Fill in the positive rate / primary metric for your problem type -->

**Classification:** 5-fold StratifiedKFold, `class_weight='balanced'` applied
where supported to account for the [X]% positive rate. Primary metric: ROC-AUC.

**Regression:** 5-fold KFold. Primary metric: R².

Preprocessor built unfitted, fit only inside each CV fold — no test set leakage.

- **Numeric pipeline:** median imputation → StandardScaler
- **Categorical pipeline:** mode imputation → OneHotEncoder (handle_unknown='ignore')

In [ ]:
preprocessor = build_preprocessor(numeric_in_X, categorical_in_X)
pipelines    = build_pipelines(preprocessor, problem_type=PROBLEM_TYPE)

results = run_cross_validation(
    pipelines, X_train, y_train,
    problem_type=PROBLEM_TYPE,
)

### 3.3 Model Selection

**Winner: [TODO: model name]**

<!-- TODO: Justify using the CV output. Apply the 2x-std rule: if models are
within 2x std of each other on the primary metric, prefer the simpler one.
State what you gain from this choice. -->

### 3.4 Hyperparameter Tuning

In [ ]:
WINNER = '[LogisticRegression / RandomForest / GradientBoosting / Ridge / ...]'

# LogisticRegression:
# param_grid = {'model__C': [0.001, 0.01, 0.1, 1.0, 10.0],
#               'model__penalty': ['l1', 'l2'], 'model__solver': ['liblinear']}
#
# RandomForest:
# param_grid = {'model__n_estimators': [100, 200, 300],
#               'model__max_depth': [3, 5, 10, None],
#               'model__min_samples_leaf': [1, 2, 5]}
#
# GradientBoosting:
# param_grid = {'model__n_estimators': [100, 200],
#               'model__learning_rate': [0.01, 0.05, 0.1],
#               'model__max_depth': [3, 5]}
#
# Ridge (regression):
# param_grid = {'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}

param_grid = {}  # TODO: replace with above

tuned_pipeline, search = tune_model(
    pipeline=pipelines[WINNER],
    X_train=X_train,
    y_train=y_train,
    param_grid=param_grid,
    problem_type=PROBLEM_TYPE,
    search_type='grid',  # 'random' for larger grids
)

print(f"Best parameters: {search.best_params_}")
print(f"Best CV score:   {search.best_score_:.4f}")

### 3.5 Feature Importance

<!-- TODO: Choose the right method:
LogisticRegression → coef_ (log-odds for classification, direct units for regression)
RandomForest / GradientBoosting → feature_importances_ -->

In [ ]:
from sklearn.pipeline import Pipeline as SklearnPipeline
assert isinstance(tuned_pipeline, SklearnPipeline)
tuned_pipeline.fit(X_train, y_train)

model = tuned_pipeline.named_steps['model']
prep  = tuned_pipeline.named_steps['preprocessor']

try:
    ohe_names = (prep.named_transformers_['cat']
                     .named_steps['onehot']
                     .get_feature_names_out(categorical_in_X).tolist())
except Exception:
    ohe_names = []

feature_names = numeric_in_X + ohe_names

# LogisticRegression — coefficients
# coef_series = pd.Series(model.coef_[0], index=feature_names)
# top_pos = coef_series.nlargest(10)
# top_neg = coef_series.nsmallest(10)
# print("Top positive coefficients:"); print(top_pos.round(3).to_string())
# print("\nTop negative coefficients:"); print(top_neg.round(3).to_string())

# RandomForest / GradientBoosting — importances
# importances = pd.Series(model.feature_importances_, index=feature_names)
# top15 = importances.nlargest(15).sort_values()
# top15.plot(kind='barh', figsize=(10, 6), color='steelblue')
# plt.title('Feature Importances (Top 15)'); plt.tight_layout(); plt.show()

pass  # remove when filled in

---
## 4. Evaluation & Interpretation

### 4.1 Final Test Set Evaluation

The test set was locked in Section 3.1. This is its one use.

In [ ]:
metrics, final_pipeline = evaluate_final_model(
    tuned_pipeline, X_train, y_train, X_test, y_test,
    problem_type=PROBLEM_TYPE,
)

### 4.2 Diagnostic Plots

<!-- TODO: Classification → threshold analysis (PR + ROC curves)
     Regression → residual analysis -->

In [ ]:
# ── Classification ─────────────────────────────────────────────────────────────
# from sklearn.metrics import precision_recall_curve, roc_curve, auc
# y_proba = final_pipeline.predict_proba(X_test)[:, 1]
# prec, rec, pr_t = precision_recall_curve(y_test, y_proba)
# fpr, tpr, _    = roc_curve(y_test, y_proba)
# fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# axes[0].plot(rec, prec, color='steelblue', lw=2)
# axes[0].axhline(y_test.mean(), color='gray', linestyle='--', label='No-skill')
# axes[0].set(xlabel='Recall', ylabel='Precision',
#             title=f'PR Curve (AUC={auc(rec,prec):.3f})'); axes[0].legend()
# axes[1].plot(fpr, tpr, color='steelblue', lw=2); axes[1].plot([0,1],[0,1],'k--')
# axes[1].set(xlabel='FPR', ylabel='TPR', title=f'ROC (AUC={auc(fpr,tpr):.3f})')
# plt.tight_layout(); plt.show()
# # Recommended threshold:
# valid = [(p,r,t) for p,r,t in zip(prec,rec,pr_t) if p >= 0.70]
# if valid:
#     bp, br, bt = max(valid, key=lambda x: x[1])
#     print(f"Recommended threshold: {bt:.3f}  Precision: {bp:.3f}  Recall: {br:.3f}")

# ── Regression ──────────────────────────────────────────────────────────────────
# y_pred = final_pipeline.predict(X_test)
# resid  = y_test - y_pred
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# axes[0].scatter(y_pred, resid, alpha=0.5, s=20)
# axes[0].axhline(0, color='red', linestyle='--')
# axes[0].set(xlabel='Predicted', ylabel='Residual', title='Residuals vs Predicted')
# axes[1].hist(resid, bins=25, edgecolor='white')
# axes[1].set(xlabel='Residual', title='Residual Distribution')
# plt.tight_layout(); plt.show()

pass  # remove when filled in

### 4.3 Business Interpretation

<!-- TODO: Interpret results for a non-technical stakeholder.

Classification:
- What does this ROC-AUC mean operationally?
- What is the recommended threshold and why?
- What are the real-world consequences of the remaining errors for this org?

Regression:
- How much better is the model than predicting the mean?
- What does the RMSE mean in the units of the target?
- Where is the model weakest and what should users know about that? -->

---
## 5. Causal and Relationship Analysis

### 5.1 Explanatory Model

We fit a separate `statsmodels` model to extract p-values and interpretable
coefficients — tools for understanding *relationships*, not improving predictions.

- **Classification targets** → `fit_causal_classification()` → statsmodels Logit
- **Regression targets** → `fit_causal_regression()` → statsmodels OLS

If the one-hot encoded matrix has more columns than rows (p > n), `SelectKBest`
reduces to the k most predictive features before fitting. This affects only the
causal model — the predictive pipeline is unaffected.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from functions.fn_model_causal import fit_causal_classification, get_coefficients
# from functions.fn_model_causal import fit_causal_regression, get_coefficients

X_train_enc = pd.get_dummies(X_train, drop_first=True, dtype=int)
X_train_enc = X_train_enc.apply(pd.to_numeric, errors='coerce').fillna(0)

n_rows, n_cols = X_train_enc.shape
print(f"Encoded matrix: {n_rows} rows × {n_cols} columns")

if n_cols >= n_rows:
    k = min(10, n_rows - 5)
    # Classification: f_classif | Regression: f_regression
    selector = SelectKBest(score_func=f_classif, k=k)
    selector.fit(X_train_enc, y_train.astype(int))
    top_cols = X_train_enc.columns[selector.get_support()]
    X_causal = X_train_enc[top_cols]
    print(f"Reduced to {k} features: {list(top_cols)}")
else:
    X_causal = X_train_enc
    print("Matrix safe for statsmodels without reduction.")

In [ ]:
# Classification:
causal_results = fit_causal_classification(X_causal, y_train.astype(int))
# Regression:
# causal_results = fit_causal_regression(X_causal, y_train)

print(causal_results.summary())

In [ ]:
coef_df = get_coefficients(causal_results, model_type='logistic')  # or 'linear'

print("Significant features (p < 0.05):")
sig = coef_df[coef_df['p_value'] < 0.05].sort_values('coefficient', ascending=False)
print(sig[['feature', 'coefficient', 'p_value', 'significant']].to_string(index=False))

### 5.2 Relationship Interpretation

<!-- TODO:
1. Group significant features into 2-3 thematic clusters
2. Explain what each cluster suggests and whether it aligns with domain knowledge
3. Name at least one specific confounder that prevents causal interpretation
4. State one actionable correlation-based insight for the organization
5. Close with: "correlation is not causation" and what it would take to establish it -->

---
## 6. Deployment

The trained pipeline is saved as a `.pkl` file. See `ml-pipelines/deployment-notes.md` for the full Azure Function, .NET controller, and React integration docs.

In [ ]:
os.makedirs('models', exist_ok=True)

pkl_path = save_model(
    final_pipeline,
    metrics,
    target_name=TARGET,
    output_dir='models',
)

print(f"Model saved: {pkl_path}")

---
## 7. API Response Reference

When the inference server scores a [resident / post / donor], it returns:

```json
{
  "[id_field]": "[int]",
  "[score_field]": "[float — 0 to 100]",
  "probability": "[float — 0.0 to 1.0]",
  "recommendation": "[string]",
  "model_version": "[TODO: model name]_v1",
  "predicted_at": "[ISO datetime]"
}
```

**probability** — Raw output from `pipeline.predict_proba(features)[0, 1]`.
The model's estimated probability (0.0–1.0) for the positive class.

**[score_field]** — `probability × 100`, rounded to 1 decimal. Human-readable
0–100 scale.

**recommendation** — Hardcoded string from threshold logic:
- Score ≥ [TODO: high threshold]: `"[TODO: high recommendation]"`
- Score ≥ [TODO: mid threshold]:  `"[TODO: mid recommendation]"`
- Score < [TODO: mid threshold]:  `"[TODO: low recommendation]"`

---
### Endpoint Function (`endpoints.py`)

Add this function to `ml-pipelines/inference/endpoints.py`:

```python
def [TODO: function_name](id_value: int, features: dict, pipeline) -> dict:
    """
    [TODO: one sentence description]
    Model: [TODO: target_name].pkl
    """
    features_df = pd.DataFrame([features])
    proba = float(pipeline.predict_proba(features_df)[0, 1])
    score = round(proba * 100, 1)

    if score >= [TODO: high threshold]:
        recommendation = "[TODO: high recommendation]"
    elif score >= [TODO: mid threshold]:
        recommendation = "[TODO: mid recommendation]"
    else:
        recommendation = "[TODO: low recommendation]"

    return {
        "[id_field]":     id_value,
        "[score_field]":  score,
        "probability":    round(proba, 4),
        "recommendation": recommendation,
        "model_version":  "[TODO: target_name]_v1",
        "predicted_at":   datetime.now(timezone.utc).isoformat(),
    }
```

---
### Route to Add (`server.py`)

```python
# Request / Response models
class [TODO: Name]Request(BaseModel):
    [TODO: id_field]: int
    features: Dict[str, Any]

class [TODO: Name]Response(BaseModel):
    [TODO: id_field]:    int
    [TODO: score_field]: float
    probability:         float
    recommendation:      str
    model_version:       str
    predicted_at:        str

# Route
@app.post("/predict/[TODO: route-name]", response_model=[TODO: Name]Response)
def predict_[TODO: function_name](request: [TODO: Name]Request):
    try:
        pipeline = load_model("[TODO: target_name]")
    except FileNotFoundError as e:
        raise HTTPException(status_code=503, detail=str(e))
    try:
        return [TODO: function_name](
            [TODO: id_field]=request.[TODO: id_field],
            features=request.features,
            pipeline=pipeline,
        )
    except Exception as e:
        log.error(f"Prediction failed: {e}")
        raise HTTPException(status_code=500, detail=f"Prediction failed: {e}")
```

---
*Hearth Haven — IS 455 INTEX Pipeline*